In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
file_path = r"C:\Users\LATO\Documents\Customer Churn\data\customer-churn-data.csv"

df = pd.read_csv(file_path)

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

# Check the target variable
print("Dataset shape:", df.shape)
print("\nChurn counts:")
print(df["Churn"].value_counts())

print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nData preparation completed!")

Dataset shape: (7043, 21)

Churn counts:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Missing values:
TotalCharges    11
dtype: int64

Data preparation completed!


In [3]:
df["ChurnBinary"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print(df[["Churn", "ChurnBinary"]].head())
print("\nChurn rate:", round(df["ChurnBinary"].mean() * 100, 2), "%")

  Churn  ChurnBinary
0    No            0
1    No            0
2   Yes            1
3    No            0
4   Yes            1

Churn rate: 26.54 %


In [4]:
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]

summary_stats = (
    df.groupby("Churn")[numeric_features]
      .agg(["count", "mean", "median", "std", "min", "max"])
      .round(2)
)

summary_stats

tenure                              MonthlyCharges                       \
       count   mean median    std min max          count   mean median    std   
Churn                                                                           
No      5174  37.57   38.0  24.11   0  72           5174  61.27  64.43  31.09   
Yes     1869  17.98   10.0  19.53   1  72           1869  74.44  79.65  24.67   

                     TotalCharges                                             
         min     max        count     mean   median      std    min      max  
Churn                                                                         
No     18.25  118.75         5163  2555.34  1683.60  2329.46  18.80  8672.45  
Yes    18.85  118.35         1869  1531.80   703.55  1890.82  18.85  8684.80

In [5]:
mean_comparison = (
    df.groupby("Churn")[numeric_features]
      .mean()
      .T
)

mean_comparison["Difference (Yes - No)"] = (
    mean_comparison["Yes"] - mean_comparison["No"]
)

mean_comparison.round(2)

Churn,No,Yes,Difference (Yes - No)
tenure,37.57,17.98,-19.59
MonthlyCharges,61.27,74.44,13.18
TotalCharges,2555.34,1531.80,-1023.55


In [6]:
churn_summary = pd.DataFrame({
    "Customer Count": df["Churn"].value_counts(),
    "Percentage": (
        df["Churn"].value_counts(normalize=True) * 100
    ).round(2)
})

churn_summary

,Customer Count,Percentage
Churn,,
No,5174,73.46
Yes,1869,26.54


In [7]:
from scipy.stats import chi2_contingency

# Create contingency table
contract_table = pd.crosstab(df["Contract"], df["Churn"])

print("Observed frequency table:")
display(contract_table)

# Chi-square test
chi2, p_value, dof, expected = chi2_contingency(contract_table)

print(f"Chi-square statistic: {chi2:.3f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p_value:.6e}")

if p_value < 0.05:
    print("Conclusion: Contract type is significantly associated with churn.")
else:
    print("Conclusion: No statistically significant association detected.")

Observed frequency table:


Churn,No,Yes
Contract,,
Month-to-month,2220,1655
One year,1307,166
Two year,1647,48


Chi-square statistic: 1184.597
Degrees of freedom: 2
P-value: 5.863038e-258
Conclusion: Contract type is significantly associated with churn.


In [8]:
import numpy as np

n = contract_table.to_numpy().sum()
min_dimension = min(contract_table.shape[0] - 1,
                    contract_table.shape[1] - 1)

cramers_v = np.sqrt(chi2 / (n * min_dimension))

print(f"Cramér's V: {cramers_v:.3f}")

Cramér's V: 0.410


In [9]:
from scipy.stats import chi2_contingency
import numpy as np
import pandas as pd

categorical_features = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

results = []

for feature in categorical_features:
    table = pd.crosstab(df[feature], df["Churn"])

    chi2, p_value, dof, expected = chi2_contingency(table)

    n = table.to_numpy().sum()
    min_dimension = min(table.shape[0] - 1, table.shape[1] - 1)
    cramers_v = np.sqrt(chi2 / (n * min_dimension))

    results.append({
        "Feature": feature,
        "Chi-square": chi2,
        "Degrees of Freedom": dof,
        "P-value": p_value,
        "Cramers V": cramers_v
    })

chi_square_results = pd.DataFrame(results)

chi_square_results["Significant (p < 0.05)"] = (
    chi_square_results["P-value"] < 0.05
)

chi_square_results = chi_square_results.sort_values(
    "Cramers V", ascending=False
)

chi_square_results.round(4)

,Feature,Chi-square,Degrees of Freedom,P-value,Cramers V,Significant (p < 0.05)
13,Contract,1184.5966,2,0.0000,0.4101,True
7,OnlineSecurity,849.9990,2,0.0000,0.3474,True
10,TechSupport,828.1971,2,0.0000,0.3429,True
6,InternetService,732.3096,2,0.0000,0.3225,True
15,PaymentMethod,648.1423,3,0.0000,0.3034,True
8,OnlineBackup,601.8128,2,0.0000,0.2923,True
9,DeviceProtection,558.4194,2,0.0000,0.2816,True
12,StreamingMovies,375.6615,2,0.0000,0.2310,True
11,StreamingTV,374.2039,2,0.0000,0.2305,True
14,PaperlessBilling,258.2776,1,0.0000,0.1915,True


In [10]:
display_results = chi_square_results.copy()

display_results["P-value"] = display_results["P-value"].map(
    lambda p: f"{p:.3e}"
)

display_results["Chi-square"] = chi_square_results["Chi-square"].round(3)
display_results["Cramers V"] = chi_square_results["Cramers V"].round(3)

display(display_results.reset_index(drop=True))

,Feature,Chi-square,Degrees of Freedom,P-value,Cramers V,Significant (p < 0.05)
0,Contract,1184.597,2,5.863e-258,0.410,True
1,OnlineSecurity,849.999,2,2.661e-185,0.347,True
2,TechSupport,828.197,2,1.443e-180,0.343,True
3,InternetService,732.310,2,9.572e-160,0.322,True
4,PaymentMethod,648.142,3,3.682e-140,0.303,True
5,OnlineBackup,601.813,2,2.080e-131,0.292,True
6,DeviceProtection,558.419,2,5.505e-122,0.282,True
7,StreamingMovies,375.661,2,2.668e-82,0.231,True
8,StreamingTV,374.204,2,5.529e-82,0.231,True
9,PaperlessBilling,258.278,1,4.073e-58,0.191,True


In [11]:
output_path = (
    r"C:\Users\LATO\Documents\Customer Churn"
    r"\chi_square_results.csv"
)

chi_square_results.to_csv(output_path, index=False)

print("Results saved to:", output_path)

Results saved to: C:\Users\LATO\Documents\Customer Churn\chi_square_results.csv


In [12]:
from scipy.stats import mannwhitneyu

numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]

test_results = []

for feature in numeric_features:
    stayed = df.loc[df["Churn"] == "No", feature].dropna()
    churned = df.loc[df["Churn"] == "Yes", feature].dropna()

    u_stat, p_value = mannwhitneyu(
        stayed,
        churned,
        alternative="two-sided"
    )

    test_results.append({
        "Feature": feature,
        "Stayed Median": stayed.median(),
        "Churned Median": churned.median(),
        "U Statistic": u_stat,
        "P-value": p_value,
        "Significant (p < 0.05)": p_value < 0.05
    })

mann_whitney_results = pd.DataFrame(test_results)

mann_whitney_results.round(4)

,Feature,Stayed Median,Churned Median,U Statistic,P-value,Significant (p < 0.05)
0,tenure,38.000,10.00,7154668.0,0.0,True
1,MonthlyCharges,64.425,79.65,3667080.5,0.0,True
2,TotalCharges,1683.600,703.55,6288982.0,0.0,True


In [13]:
effect_results = []

for _, row in mann_whitney_results.iterrows():
    feature = row["Feature"]

    stayed = df.loc[df["Churn"] == "No", feature].dropna()
    churned = df.loc[df["Churn"] == "Yes", feature].dropna()

    u_stat, p_value = mannwhitneyu(
        stayed, churned, alternative="two-sided"
    )

    n_stayed = len(stayed)
    n_churned = len(churned)

    # Positive means values tend to be higher among churned customers
    rank_biserial = 1 - (2 * u_stat) / (n_stayed * n_churned)

    effect_results.append({
        "Feature": feature,
        "Rank-biserial correlation": rank_biserial,
        "P-value": p_value
    })

effect_size_results = pd.DataFrame(effect_results)

effect_size_results.round(4)

,Feature,Rank-biserial correlation,P-value
0,tenure,-0.4797,0.0
1,MonthlyCharges,0.2416,0.0
2,TotalCharges,-0.3035,0.0


In [14]:
from scipy.stats import bootstrap
import numpy as np
import pandas as pd

numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]

ci_results = []

for feature in numeric_features:
    stayed = df.loc[df["Churn"] == "No", feature].dropna().to_numpy()
    churned = df.loc[df["Churn"] == "Yes", feature].dropna().to_numpy()

    observed_difference = np.median(churned) - np.median(stayed)

    result = bootstrap(
        (churned, stayed),
        statistic=lambda x, y: np.median(x) - np.median(y),
        vectorized=False,
        n_resamples=2000,
        confidence_level=0.95,
        method="percentile",
        random_state=42
    )

    ci_results.append({
        "Feature": feature,
        "Median Difference (Yes - No)": observed_difference,
        "95% CI Lower": result.confidence_interval.low,
        "95% CI Upper": result.confidence_interval.high
    })

confidence_intervals = pd.DataFrame(ci_results).round(2)

confidence_intervals

,Feature,Median Difference (Yes - No),95% CI Lower,95% CI Upper
0,tenure,-28.00,-30.00,-26.0
1,MonthlyCharges,15.22,13.95,17.8
2,TotalCharges,-980.05,-1095.11,-860.0


In [15]:
confidence_intervals["Excludes Zero"] = (
    (confidence_intervals["95% CI Lower"] > 0) |
    (confidence_intervals["95% CI Upper"] < 0)
)

confidence_intervals

,Feature,Median Difference (Yes - No),95% CI Lower,95% CI Upper,Excludes Zero
0,tenure,-28.00,-30.00,-26.0,True
1,MonthlyCharges,15.22,13.95,17.8,True
2,TotalCharges,-980.05,-1095.11,-860.0,True


In [16]:
from scipy.stats import pearsonr, spearmanr
import pandas as pd

numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]

correlation_results = []

for i in range(len(numeric_features)):
    for j in range(i + 1, len(numeric_features)):
        feature1 = numeric_features[i]
        feature2 = numeric_features[j]

        pair = df[[feature1, feature2]].dropna()

        pearson_r, pearson_p = pearsonr(
            pair[feature1], pair[feature2]
        )

        spearman_r, spearman_p = spearmanr(
            pair[feature1], pair[feature2]
        )

        correlation_results.append({
            "Feature 1": feature1,
            "Feature 2": feature2,
            "Sample Size": len(pair),
            "Pearson r": pearson_r,
            "Pearson p-value": pearson_p,
            "Spearman rho": spearman_r,
            "Spearman p-value": spearman_p
        })

correlation_results = pd.DataFrame(correlation_results)

correlation_results.round(4)

,Feature 1,Feature 2,Sample Size,Pearson r,Pearson p-value,Spearman rho,Spearman p-value
0,tenure,MonthlyCharges,7043,0.2479,0.0,0.2764,0.0
1,tenure,TotalCharges,7032,0.8259,0.0,0.8892,0.0
2,MonthlyCharges,TotalCharges,7032,0.6511,0.0,0.6380,0.0


In [17]:
from statsmodels.stats.multitest import multipletests

chi_square_results["P-value adjusted (BH)"] = multipletests(
    chi_square_results["P-value"],
    alpha=0.05,
    method="fdr_bh"
)[1]

chi_square_results["Significant after correction"] = (
    chi_square_results["P-value adjusted (BH)"] < 0.05
)

chi_square_results[
    [
        "Feature",
        "P-value",
        "P-value adjusted (BH)",
        "Cramers V",
        "Significant after correction"
    ]
].sort_values("P-value adjusted (BH)").round(6)

,Feature,P-value,P-value adjusted (BH),Cramers V,Significant after correction
13,Contract,0.000000,0.000000,0.410116,True
7,OnlineSecurity,0.000000,0.000000,0.347400,True
10,TechSupport,0.000000,0.000000,0.342916,True
6,InternetService,0.000000,0.000000,0.322455,True
15,PaymentMethod,0.000000,0.000000,0.303359,True
8,OnlineBackup,0.000000,0.000000,0.292316,True
9,DeviceProtection,0.000000,0.000000,0.281580,True
12,StreamingMovies,0.000000,0.000000,0.230951,True
11,StreamingTV,0.000000,0.000000,0.230502,True
14,PaperlessBilling,0.000000,0.000000,0.191498,True


In [18]:
print(
    "Significant categorical features after BH correction:",
    chi_square_results["Significant after correction"].sum(),
    "out of",
    len(chi_square_results)
)

Significant categorical features after BH correction: 14 out of 16


In [19]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

In [20]:
model_data = df[
    [
        "ChurnBinary",
        "tenure",
        "MonthlyCharges",
        "Contract",
        "InternetService",
        "PaymentMethod",
        "TechSupport",
        "OnlineSecurity",
        "SeniorCitizen",
        "Partner",
        "Dependents"
    ]
].dropna().copy()

formula = (
    "ChurnBinary ~ tenure + MonthlyCharges + "
    "C(Contract) + C(InternetService) + "
    "C(PaymentMethod) + C(TechSupport) + "
    "C(OnlineSecurity) + SeniorCitizen + "
    "C(Partner) + C(Dependents)"
)

logit_model = smf.logit(
    formula=formula,
    data=model_data
).fit()

print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.421246
         Iterations 8


LinAlgError: Singular matrix

In [21]:
# Check whether any predictor has only one unique value
predictors = [
    "tenure",
    "MonthlyCharges",
    "Contract",
    "InternetService",
    "PaymentMethod",
    "TechSupport",
    "OnlineSecurity",
    "SeniorCitizen",
    "Partner",
    "Dependents"
]

for col in predictors:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))


tenure:
tenure
1     613
72    362
2     238
3     200
4     176
     ... 
28     57
39     56
44     51
36     50
0      11
Name: count, Length: 73, dtype: int64

MonthlyCharges:
MonthlyCharges
20.05     61
19.85     45
19.95     44
19.90     44
20.00     43
          ..
56.85      1
101.70     1
48.40      1
108.35     1
72.00      1
Name: count, Length: 1585, dtype: int64

Contract:
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

InternetService:
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

PaymentMethod:
PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64

TechSupport:
TechSupport
No                     3473
Yes                    2044
No internet service    1526
Name: count, dtype: int64

OnlineSecurity:
OnlineSecurity
No                     349

In [22]:
import patsy
import numpy as np

y, X = patsy.dmatrices(
    formula,
    data=model_data,
    return_type="dataframe"
)

print("Design matrix shape:", X.shape)
print("Matrix rank:", np.linalg.matrix_rank(X))
print("Number of columns:", X.shape[1])
print("Rank deficient:", np.linalg.matrix_rank(X) < X.shape[1])

Design matrix shape: (7043, 17)
Matrix rank: 15
Number of columns: 17
Rank deficient: True


In [23]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

formula_v2 = (
    "ChurnBinary ~ tenure + MonthlyCharges + "
    "C(Contract) + C(PaymentMethod) + "
    "C(TechSupport) + C(OnlineSecurity) + "
    "SeniorCitizen + C(Partner) + C(Dependents)"
)

logit_model = smf.logit(
    formula=formula_v2,
    data=model_data
).fit()

print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.422231
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:            ChurnBinary   No. Observations:                 7043
Model:                          Logit   Df Residuals:                     7029
Method:                           MLE   Df Model:                           13
Date:                Thu, 29 Oct 2026   Pseudo R-squ.:                  0.2703
Time:                        10:58:51   Log-Likelihood:                -2973.8
converged:                       True   LL-Null:                       -4075.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                  coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------
Intercept                                 

In [24]:
# Restrict this secondary model to customers with internet service
internet_data = model_data[
    model_data["TechSupport"] != "No internet service"
].copy()

formula_v3 = (
    "ChurnBinary ~ tenure + MonthlyCharges + "
    "C(Contract) + C(PaymentMethod) + "
    "C(TechSupport) + C(OnlineSecurity) + "
    "SeniorCitizen + C(Partner) + C(Dependents)"
)

logit_internet = smf.logit(
    formula=formula_v3,
    data=internet_data
).fit()

print(logit_internet.summary())

Optimization terminated successfully.
         Current function value: 0.480688
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:            ChurnBinary   No. Observations:                 5517
Model:                          Logit   Df Residuals:                     5504
Method:                           MLE   Df Model:                           12
Date:                Thu, 29 Oct 2026   Pseudo R-squ.:                  0.2316
Time:                        10:59:31   Log-Likelihood:                -2652.0
converged:                       True   LL-Null:                       -3451.3
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                  coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------
Intercept                                 

In [25]:
# Calculate adjusted odds ratios and 95% confidence intervals
params = logit_internet.params
conf = logit_internet.conf_int()

odds_results = pd.DataFrame({
    "Coefficient": params,
    "Odds Ratio": np.exp(params),
    "CI Lower": np.exp(conf[0]),
    "CI Upper": np.exp(conf[1]),
    "P-value": logit_internet.pvalues
})

# Exclude the intercept for easier interpretation
odds_results = odds_results.drop(index="Intercept")

odds_results["Significant"] = odds_results["P-value"] < 0.05

odds_results = odds_results.sort_values("P-value")

display(odds_results.round(4))

,Coefficient,Odds Ratio,CI Lower,CI Upper,P-value,Significant
tenure,-0.0320,0.9685,0.9642,0.9728,0.0000,True
MonthlyCharges,0.0217,1.0220,1.0183,1.0256,0.0000,True
C(Contract)[T.Two year],-1.2890,0.2756,0.1917,0.3960,0.0000,True
C(OnlineSecurity)[T.Yes],-0.5697,0.5657,0.4811,0.6651,0.0000,True
C(TechSupport)[T.Yes],-0.5168,0.5964,0.5063,0.7026,0.0000,True
C(Contract)[T.One year],-0.6383,0.5282,0.4257,0.6554,0.0000,True
C(PaymentMethod)[T.Electronic check],0.4192,1.5208,1.2596,1.8361,0.0000,True
SeniorCitizen,0.2793,1.3221,1.1206,1.5600,0.0009,True
C(Dependents)[T.Yes],-0.1910,0.8261,0.6879,0.9921,0.0409,True
C(PaymentMethod)[T.Mailed check],-0.0655,0.9366,0.7376,1.1893,0.5909,False


In [26]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import patsy
import pandas as pd

y, X = patsy.dmatrices(
    formula_v3,
    data=internet_data,
    return_type="dataframe"
)

vif_results = pd.DataFrame({
    "Feature": X.columns,
    "VIF": [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]
})

vif_results = vif_results[
    vif_results["Feature"] != "Intercept"
].sort_values("VIF", ascending=False)

display(vif_results.round(3))

,Feature,VIF
10,tenure,2.599
2,C(Contract)[T.Two year],2.371
4,C(PaymentMethod)[T.Electronic check],1.933
5,C(PaymentMethod)[T.Mailed check],1.612
1,C(Contract)[T.One year],1.574
3,C(PaymentMethod)[T.Credit card (automatic)],1.557
8,C(Partner)[T.Yes],1.413
6,C(TechSupport)[T.Yes],1.334
9,C(Dependents)[T.Yes],1.315
7,C(OnlineSecurity)[T.Yes],1.264


# Statistical Analysis — Customer Churn Prediction

## 1. Objective
The objective was to identify customer characteristics associated with churn using descriptive statistics, hypothesis testing, correlation analysis, and logistic regression.

## 2. Key Findings

- **Churn rate:** 26.54% of customers discontinued the service.
- **Contract type:** Month-to-month customers had a 42.71% churn rate, compared with 11.27% for one-year contracts and 2.83% for two-year contracts.
- **Tenure:** Churned customers had a median tenure of 10 months, compared with 38 months for retained customers.
- **Monthly charges:** Churned customers had a median monthly charge of 79.65, compared with 64.43 for retained customers.
- **Support and security:** Customers with technical support and online security showed lower churn rates.
- **Payment method:** Electronic-check customers had a substantially higher churn rate than customers using other payment methods.

## 3. Hypothesis Testing

Chi-square tests found significant associations between churn and 14 of 16 categorical features after Benjamini–Hochberg false discovery rate correction. Contract type showed the strongest association, with Cramér's V of approximately 0.410.

Mann–Whitney U tests found statistically significant differences in tenure, monthly charges, and total charges between churned and retained customers. Bootstrap confidence intervals for median differences excluded zero.

Correlation analysis found a strong positive relationship between tenure and total charges (Pearson r = 0.826). This is reasonable because total charges accumulate over time.

## 4. Logistic Regression

A multivariable logistic regression was fitted to 5,517 customers with internet service. The model was statistically significant overall (likelihood-ratio p < 0.001), with a pseudo-R² of 0.2316.

After adjusting for the other included variables:

- Longer tenure was associated with lower churn odds.
- Higher monthly charges were associated with higher churn odds.
- One-year and two-year contracts were associated with lower churn odds than month-to-month contracts.
- Technical support and online security were associated with lower churn odds.
- Electronic-check payment and senior-citizen status were associated with higher churn odds.
- Dependents were associated with modestly lower churn odds.

The VIF values ranged from 1.117 to 2.599, providing no indication of serious multicollinearity among the predictors in this model.

## 5. Business Recommendations

1. Prioritize retention efforts for month-to-month customers.
2. Engage new customers early, especially during their first year.
3. Investigate customer concerns related to monthly charges and perceived value.
4. Promote the benefits of technical support and online security.
5. Investigate why electronic-check customers show elevated churn.
6. Monitor churn patterns across customer segments instead of assuming one strategy fits everyone.

## 6. Limitations

These analyses identify associations rather than causal effects. The logistic regression was restricted to customers with internet service, so its results should not automatically be generalized to customers without internet service. Statistical significance does not necessarily imply practical importance. Predictive performance must be evaluated separately using held-out data and suitable metrics.

## Conclusion

Contract type, tenure, monthly charges, support services, security services, and payment method are useful factors for understanding customer churn. These findings provide a statistical foundation for building and evaluating a customer churn prediction model.
